# FedAvg Server

> The core abstraction for `fedavg` server.

In [ ]:
#| default_exp servers.fedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.servers.base_server import BaseServer


<torch._C.Generator>

`state = {'model': model, 'optimizer': None, 'criterion': criterion, 't': t, 'h': None, 'h_c': None, "pers_model": None}`


State Manager Requirements:
- For Prsonalized FL algos, the client's model must be stored throuout the communication rounds.
- Some optimizers (e.g., Adam) have internal states that must be stored and sent back to the client at each round.
- Other information that the client may need to store and retrieve across rounds (`feature anchor: h`, `coalition's anchor: h_c`, `personalized model: pers_model`).

In [ ]:
#| export
import torch

class StateManager:
    def __init__(self):
        self.registry = {} 

    def set_state(self, client_id, state_dict):
        """
        Takes a dictionary of tensors/objects and moves them to CPU.
        Works for weights, optimizer states, anchors (h), etc.
        """
        if client_id not in self.registry:
            self.registry[client_id] = {}

        for key, value in state_dict.items():
            # Deep clone and move to CPU if it's a torch object
            if isinstance(value, torch.Tensor):
                self.registry[client_id][key] = value.detach().cpu().clone()
            elif isinstance(value, dict):
                # Handle nested dicts (like optimizer.state_dict())
                self.registry[client_id][key] = self._to_cpu(value)
            else:
                self.registry[client_id][key] = value

    def get_state(self, client_id):
        return self.registry.get(client_id, {})

    def _to_cpu(self, obj):
        """Recursively moves tensors in a nested structure to CPU."""
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().clone()
        elif isinstance(obj, dict):
            return {k: self._to_cpu(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._to_cpu(v) for v in obj]
        return obj

In [ ]:
#| hide
import torch
ad = torch.optim.Adam([torch.zeros(1)], lr=0.001)
ad_state_dict = ad.state_dict()
print(ad_state_dict)
print(ad_state_dict.items())
# cpu_optimizer_dict = {k: v.cpu().clone() for k, v in ad_state_dict}

{'state': {}, 'param_groups': [{'lr': 0.001, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0, 'amsgrad': False, 'maximize': False, 'foreach': None, 'capturable': False, 'differentiable': False, 'fused': None, 'decoupled_weight_decay': False, 'params': [0]}]}
dict_items([('state', {}), ('param_groups', [{'lr': 0.001, 'betas': (0.9, 0.999), 'eps': 1e-08, 'weight_decay': 0, 'amsgrad': False, 'maximize': False, 'foreach': None, 'capturable': False, 'differentiable': False, 'fused': None, 'decoupled_weight_decay': False, 'params': [0]}])])


In [ ]:
#| export
class DiskStateManager(StateManager):
    def __init__(self, state_dir, mode="global"):
        super().__init__(mode)
        self.state_dir = state_dir
        os.makedirs(self.state_dir, exist_ok=True)

    def _get_path(self, client_id):
        if self.mode == "global":
            return os.path.join(self.state_dir, "global_model.pth")
        else:
            return os.path.join(self.state_dir, f"client_{client_id}_model.pth")

    def get_model(self, client_id):
        path = self._get_path(client_id)
        if os.path.exists(path):
            return torch.load(path, map_location='cpu')
        return None

    def set_model(self, client_id, state_dict):
        path = self._get_path(client_id)
        torch.save(state_dict, path)

## ServerFedAvg

In [ ]:
#| export
class ServerFedAvg(BaseServer):
    def __init__(self,
                 cfg,
                 client_fn,
                 client_selector,
                 client_cls,
                 criterion,
                 writer,
                 create_model_fn= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_fn, client_selector, client_cls, criterion, writer, create_model_fn, **kwargs)

We need to make sure that all clients start from the same initial model at round 0. So, we need to save the initial model once at the server side and load it at each client during its initialization.

### Aggregation


The aggegation for `fedavg` is defined as:

$$m_t \leftarrow \sum_{k \in S_t} n_k$$
$$W_{global}^{(t + 1)} \leftarrow \sum_{k \in S_t} \frac{n_k}{m_t} w_k^{(t + 1)}$$

where $S_t$ is the set of active clients at communication round $t$, $n_k$ is the number of samples at client $k$, and $w_k^{(t)}$ is the model parameters of client $k$ after local training at round $t$.

In [ ]:
#| export
@patch
def aggregate(self: ServerFedAvg, lst_active_ids, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)

        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'model': self.model.state_dict(),
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()